In [1]:
import sys
import pandas as pd
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [ ]:
from src.data_processing import build_player_dataset

In [ ]:
brunson_dataset = build_player_dataset(
    player_id=1628973,
    season="2025-26"
)

brunson_dataset.head()

In [ ]:
from nba_api.stats.static import players

players.find_players_by_full_name("Nikola Jokic")

In [ ]:
jokic_dataset = build_player_dataset(
    player_id=203999,
    season="2025-26"
)

jokic_dataset.head()

In [ ]:
combined = pd.concat([brunson_dataset, jokic_dataset])

combined.head(10)

## Create combined dataset for 2 players (Jalen Brunson and Nikola Jokic)...

In [7]:
test_players = [
    {"name": "Jalen Brunson", "id": 1628973},
    {"name": "Nikola Jokic", "id": 203999},
]

In [24]:
player_datasets = []

for player in test_players:
    print(f"Building dataset for {player['name']}...")

    player_data = build_player_dataset(
        player_id=player["id"],
        season="2025-26"
    )

    player_data["PLAYER_NAME"] = player["name"]

    player_datasets.append(player_data)

    combined_dataset = pd.concat(
    player_datasets,
    ignore_index=True
)
    
combined_dataset[
    [
        "PLAYER_NAME",
        "PLAYER_ID",
        "GAME_DATE",
        "MATCHUP",
        "PTS",
        "REB",
        "AST",
        "PTS_LAST_5_AVG",
    ]
].head(10)

Building dataset for Jalen Brunson...
Building dataset for Nikola Jokic...


,PLAYER_NAME,PLAYER_ID,GAME_DATE,MATCHUP,PTS,REB,AST,PTS_LAST_5_AVG
0,Jalen Brunson,1628973,2025-10-22,NYK vs. CLE,23,4,5,NaN
1,Jalen Brunson,1628973,2025-10-24,NYK vs. BOS,31,3,5,23.000000
2,Jalen Brunson,1628973,2025-10-26,NYK @ MIA,37,5,7,27.000000
3,Jalen Brunson,1628973,2025-10-28,NYK @ MIL,36,4,3,30.333333
4,Jalen Brunson,1628973,2025-10-31,NYK @ CHI,29,2,7,31.750000
5,Jalen Brunson,1628973,2025-11-02,NYK vs. CHI,31,5,3,31.200000
6,Jalen Brunson,1628973,2025-11-03,NYK vs. WAS,16,1,9,32.800000
7,Jalen Brunson,1628973,2025-11-05,NYK vs. MIN,23,7,10,29.800000
8,Jalen Brunson,1628973,2025-11-09,NYK vs. BKN,19,0,7,27.000000
9,Jalen Brunson,1628973,2025-11-11,NYK vs. MEM,32,5,10,23.600000


In [ ]:
combined_dataset["PLAYER_NAME"].value_counts()

## Create full dataset for all players...

In [ ]:
from nba_api.stats.static import players

all_players = players.get_players()

active_players = [
    player for player in all_players
    if player["is_active"]
]

len(active_players)

In [ ]:
import time

player_datasets = []
failed_players = []

for index, player in enumerate(active_players, start=1):
    try:
        player_data = build_player_dataset(
            player_id=player["id"],
            season="2025-26"
        )

        if player_data.empty:
            print(f"No games found for {player['full_name']}")
            continue

        player_data["PLAYER_NAME"] = player["full_name"]

        player_datasets.append(player_data)

        if index % 25 == 0 or index == len(active_players):
            print(
                f"Processed {index}/{len(active_players)} players..."
            )

        time.sleep(0.6)

    except Exception as error:
        print(f"Failed for {player['full_name']}: {error}")

        failed_players.append({
            "PLAYER_NAME": player["full_name"],
            "PLAYER_ID": player["id"],
            "ERROR": str(error)
        })

print(f"\nSuccessfully processed: {len(player_datasets)} players")
print(f"Failed: {len(failed_players)} players")

In [ ]:
print("Dataset shape:", combined_dataset.shape)
print("Players included:", combined_dataset["PLAYER_NAME"].nunique())
print("Date range:", combined_dataset["GAME_DATE"].min(), "to", combined_dataset["GAME_DATE"].max())
print("Duplicate rows:", combined_dataset.duplicated(
    subset=["PLAYER_ID", "Game_ID"]
).sum())

In [13]:
full_dataset = pd.concat(
    player_datasets,
    ignore_index=True
)

In [ ]:
print("Dataset shape:", full_dataset.shape)
print("Players included:", full_dataset["PLAYER_NAME"].nunique())
print(
    "Date range:",
    full_dataset["GAME_DATE"].min(),
    "to",
    full_dataset["GAME_DATE"].max()
)
print(
    "Duplicate rows:",
    full_dataset.duplicated(
        subset=["Player_ID", "Game_ID"]
    ).sum()
)

In [18]:
feature_columns = [
    "IS_HOME",
    "REST_DAYS",
    "GAMES_PLAYED",

    "PTS_LAST_GAME",
    "PTS_LAST_5_AVG",
    "PTS_LAST_10_AVG",
    "PTS_SEASON_AVG",
    "PTS_STD_LAST_5",
    "PTS_STD_LAST_10",

    "REB_LAST_GAME",
    "REB_LAST_5_AVG",
    "REB_LAST_10_AVG",
    "REB_SEASON_AVG",
    "REB_STD_LAST_5",
    "REB_STD_LAST_10",

    "AST_LAST_GAME",
    "AST_LAST_5_AVG",
    "AST_LAST_10_AVG",
    "AST_SEASON_AVG",
    "AST_STD_LAST_5",
    "AST_STD_LAST_10",

    "MIN_LAST_GAME",
    "MIN_LAST_5_AVG",
    "MIN_LAST_10_AVG",
    "MIN_SEASON_AVG",
    "MIN_STD_LAST_5",
    "MIN_STD_LAST_10",

    "FGA_LAST_GAME",
    "FGA_LAST_5_AVG",
    "FGA_LAST_10_AVG",
    "FGA_SEASON_AVG",
    "FGA_STD_LAST_5",
    "FGA_STD_LAST_10",

    "FG3A_LAST_GAME",
    "FG3A_LAST_5_AVG",
    "FG3A_LAST_10_AVG",
    "FG3A_SEASON_AVG",
    "FG3A_STD_LAST_5",
    "FG3A_STD_LAST_10",

    "FTA_LAST_GAME",
    "FTA_LAST_5_AVG",
    "FTA_LAST_10_AVG",
    "FTA_SEASON_AVG",
    "FTA_STD_LAST_5",
    "FTA_STD_LAST_10",
]

In [25]:
print("Dataset shape:", full_dataset.shape)
print("Players included:", full_dataset["PLAYER_NAME"].nunique())
print(
    "Date range:",
    full_dataset["GAME_DATE"].min(),
    "to",
    full_dataset["GAME_DATE"].max()
)
print(
    "Duplicate rows:",
    full_dataset.duplicated(
        subset=["Player_ID", "Game_ID"]
    ).sum()
)

Dataset shape: (22926, 75)
Players included: 464
Date range: 2025-10-21 00:00:00 to 2026-04-12 00:00:00
Duplicate rows: 0


In [17]:
full_dataset.to_csv(
    "../data/processed/full_dataset.csv",
    index=False
)

In [20]:
model_dataset_v2 = full_dataset.dropna(
    subset=feature_columns
).copy()

In [26]:
print("V2 model dataset shape:", model_dataset_v2.shape)

print(
    "Total missing feature values:",
    model_dataset_v2[feature_columns].isna().sum().sum()
)

print(
    "Players included:",
    model_dataset_v2["PLAYER_NAME"].nunique()
)

V2 model dataset shape: (22000, 75)
Total missing feature values: 0
Players included: 459


In [22]:
model_dataset_v2.to_csv(
    "../data/processed/model_dataset_v2.csv",
    index=False
)